In [15]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import glob
import seaborn as sns

## 1. Bright条件: ログ行数チェック（S01～S19）

In [ ]:
bright_results = []

for s in range(1, 20):
    subj = f"S{s:02d}"
    for seg in range(0, 10):
        csv_path = f"../../log/Bright/{subj}/{subj}_{seg}.csv"
        
        if not os.path.exists(csv_path):
            continue
        
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
            n_rows = len(df)
            
            if n_rows != 48:
                bright_results.append({"Condition": "Bright", "Subject": subj, "Segment": seg, "Rows": n_rows})
        except Exception as e:
            bright_results.append({"Condition": "Bright", "Subject": subj, "Segment": seg, "Rows": f"エラー: {e}"})

if bright_results:
    df_bright = pd.DataFrame(bright_results)
    print(f"【Bright: 48行ではないログ】 {len(bright_results)}件")
    display(df_bright)
else:
    print("Bright: 全て48行です✓")

## 2. Dark条件: ログ行数チェック（S101～S119）

In [ ]:
dark_results = []

for s in range(101, 120):
    subj = f"S{s:03d}"
    for seg in range(0, 10):
        csv_path = f"../../log/Dark/{subj}/{subj}_{seg}.csv"
        
        if not os.path.exists(csv_path):
            continue
        
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
            n_rows = len(df)
            
            if n_rows != 48:
                dark_results.append({"Condition": "Dark", "Subject": subj, "Segment": seg, "Rows": n_rows})
        except Exception as e:
            dark_results.append({"Condition": "Dark", "Subject": subj, "Segment": seg, "Rows": f"エラー: {e}"})

if dark_results:
    df_dark = pd.DataFrame(dark_results)
    print(f"【Dark: 48行ではないログ】 {len(dark_results)}件")
    display(df_dark)
else:
    print("Dark: 全て48行です✓")

## 3. Frame_120fps間隔チェック（Bright S01～S19, seg=0のみ）

In [ ]:
interval_results = []

for s in range(1, 20):
    subj = f"S{s:02d}"
    csv_path = f"../../log/Bright/{subj}/{subj}_0.csv"
    
    if not os.path.exists(csv_path):
        continue
    
    try:
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
        
        if 'Frame_120fps' not in df.columns:
            interval_results.append({"Subject": subj, "Status": "Frame_120fps列なし", "Median": None, "Mean": None})
            continue
        
        frames = pd.to_numeric(df['Frame_120fps'], errors='coerce').dropna().values
        
        if len(frames) < 2:
            interval_results.append({"Subject": subj, "Status": f"データ不足 (n={len(frames)})", "Median": None, "Mean": None})
            continue
        
        intervals = np.diff(frames)
        median_interval = np.median(intervals)
        mean_interval = np.mean(intervals)
        
        # 300から±50以上外れているかチェック
        status = "★異常" if abs(median_interval - 300) > 50 else "正常"
        interval_results.append({
            "Subject": subj, 
            "Status": status, 
            "Median": round(median_interval, 1), 
            "Mean": round(mean_interval, 1)
        })
            
    except Exception as e:
        interval_results.append({"Subject": subj, "Status": f"エラー: {e}", "Median": None, "Mean": None})

df_intervals = pd.DataFrame(interval_results)
print("【Frame_120fps間隔チェック】")
display(df_intervals)

# 異常のみ表示
abnormal = df_intervals[df_intervals['Status'].str.contains('★', na=False)]
if len(abnormal) > 0:
    print(f"\n異常: {len(abnormal)}件")
    display(abnormal)

## 4. 総合サマリー

In [ ]:
print("="*60)
print("【総合サマリー】")
print("="*60)

total_abnormal = 0

if 'bright_results' in locals() and bright_results:
    print(f"✗ Bright: 48行ではないログ → {len(bright_results)}件")
    total_abnormal += len(bright_results)
else:
    print("✓ Bright: ログ行数OK")

if 'dark_results' in locals() and dark_results:
    print(f"✗ Dark: 48行ではないログ → {len(dark_results)}件")
    total_abnormal += len(dark_results)
else:
    print("✓ Dark: ログ行数OK")

if 'abnormal' in locals() and len(abnormal) > 0:
    print(f"✗ Frame間隔異常 → {len(abnormal)}件")
    total_abnormal += len(abnormal)
else:
    print("✓ Frame間隔OK")

print("="*60)
if total_abnormal == 0:
    print("🎉 全てのチェックをパスしました！")
else:
    print(f"⚠️  合計 {total_abnormal} 件の異常が見つかりました")

## 5. 画像確認：Pages と Task Windows の同時表示

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob

# 被験者とセグメント番号を指定
subject_id = "S01"  # 例: S01, S02, S03, S04
segment = 0         # 例: 0～9

condition = "Bright"  # または "Dark"

# ファイルパスを構築
pages_pattern = f"../../data/graphs/pages/{condition}/{subject_id}/{subject_id}_{segment}_with_emr/{subject_id}_{segment}_with_emr_page_*.png"
task_windows_pattern = f"../../data/graphs/task_windows/{condition}/{subject_id}/{subject_id}_{segment}_*.png"

pages_files = sorted(glob.glob(pages_pattern))
task_windows_files = sorted(glob.glob(task_windows_pattern))

print(f"【{condition} {subject_id} seg={segment}】")
print(f"Pages画像: {len(pages_files)}枚")
print(f"Task Windows画像: {len(task_windows_files)}枚")
print()

if pages_files or task_windows_files:
    # 全体のレイアウトを計算
    n_pages = min(len(pages_files), 3)  # 最大3枚
    n_task_windows = len(task_windows_files)
    
    # 行数を決定（Pages行 + Task Windows行）
    n_rows = (1 if n_pages > 0 else 0) + (1 if n_task_windows > 0 else 0)
    
    if n_rows > 0:
        fig = plt.figure(figsize=(24, 8 * n_rows))
        
        current_row = 1
        
        # Pages画像を上段に横並び
        if n_pages > 0:
            for i, img_path in enumerate(pages_files[:3]):
                ax = plt.subplot(n_rows, 3, i + 1)
                img = Image.open(img_path)
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(os.path.basename(img_path), fontsize=10)
            current_row += 1
        
        # Task Windows画像を下段に表示
        if n_task_windows > 0:
            for j, img_path in enumerate(task_windows_files):
                # 下段は3列全体を使用
                ax = plt.subplot(n_rows, 1, current_row + j)
                img = Image.open(img_path)
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(os.path.basename(img_path), fontsize=10)
        
        plt.suptitle(f"{condition} {subject_id} seg={segment}", fontsize=14, y=0.995)
        plt.tight_layout()
        
        # 画像を保存
        save_dir = f"../../data/graphs/page_task_combined/{condition}/{subject_id}"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/{subject_id}_{segment}_combined.png"
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ 保存: {save_path}\n")
        
        plt.show()
        plt.close()
else:
    print("⚠️  画像が見つかりません")

### 複数被験者の一括確認

In [ ]:
# 複数被験者を一括で確認（Pages + Task Windows組み合わせ、画像保存）
# Bright: S01～S19, Dark: S101～S119, セグメント0～9

# 処理リストを作成
configs = []
for s in range(1, 20):
    for seg in range(0, 10):
        configs.append(("Bright", f"S{s:02d}", seg))
for s in range(101, 120):
    for seg in range(0, 10):
        configs.append(("Dark", f"S{s:03d}", seg))

print(f"🔄 処理開始: {len(configs)}件")
print(f"  Bright: S13～S19 × seg 0～9 (70件)")
print(f"  Dark: S101～S119 × seg 0～9 (190件)")
print("=" * 80)

success_count = 0
skip_count = 0
error_count = 0

for idx, (condition, subject_id, segment) in enumerate(configs, 1):
    print(f"\n[{idx}/{len(configs)}] 処理中: {condition} {subject_id} seg={segment}")
    print("-" * 80)
    
    try:
        # ファイルパスを構築
        pages_pattern = f"../../data/graphs/pages/{condition}/{subject_id}/{subject_id}_{segment}_with_emr/{subject_id}_{segment}_with_emr_page_*.png"
        task_windows_pattern = f"../../data/graphs/task_windows/{condition}/{subject_id}/{subject_id}_{segment}_*.png"
        
        print(f"  📂 ファイル検索中...")
        pages_files = sorted(glob.glob(pages_pattern))
        task_windows_files = sorted(glob.glob(task_windows_pattern))
        
        print(f"  ├─ Pages画像: {len(pages_files)}枚")
        print(f"  └─ Task Windows画像: {len(task_windows_files)}枚")
        
        if pages_files or task_windows_files:
            print(f"  🎨 画像生成中...")
            
            # 全体のレイアウトを計算
            n_pages = min(len(pages_files), 3)  # 最大3枚
            n_task_windows = len(task_windows_files)
            
            # 行数を決定（Pages行 + Task Windows行）
            n_rows = (1 if n_pages > 0 else 0) + (1 if n_task_windows > 0 else 0)
            
            if n_rows > 0:
                fig = plt.figure(figsize=(24, 8 * n_rows))
                
                current_row = 1
                
                # Pages画像を上段に横並び
                if n_pages > 0:
                    for i, img_path in enumerate(pages_files[:3]):
                        ax = plt.subplot(n_rows, 3, i + 1)
                        img = Image.open(img_path)
                        ax.imshow(img)
                        ax.axis('off')
                        ax.set_title(os.path.basename(img_path), fontsize=10)
                    current_row += 1
                
                # Task Windows画像を下段に表示
                if n_task_windows > 0:
                    for j, img_path in enumerate(task_windows_files):
                        # 下段は3列全体を使用
                        ax = plt.subplot(n_rows, 1, current_row + j)
                        img = Image.open(img_path)
                        ax.imshow(img)
                        ax.axis('off')
                        ax.set_title(os.path.basename(img_path), fontsize=10)
                
                plt.suptitle(f"{condition} {subject_id} seg={segment}", fontsize=14, y=0.98)
                plt.tight_layout(rect=[0, 0, 1, 0.97])
                
                # 画像を保存（被験者ごとのフォルダ）
                save_dir = f"../../data/graphs/page_task_combined/{condition}/{subject_id}"
                os.makedirs(save_dir, exist_ok=True)
                save_path = f"{save_dir}/{subject_id}_{segment}_combined.png"
                
                print(f"  💾 保存中...")
                fig.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0.3)
                print(f"  ✅ 完了: {os.path.basename(save_path)}")
                
                plt.close()
                success_count += 1
        else:
            print(f"  ⚠️  画像が見つかりません（スキップ）")
            skip_count += 1
    
    except Exception as e:
        print(f"  ❌ エラー: {e}")
        error_count += 1
        continue

print("\n" + "=" * 80)
print("🎉 全処理完了")
print(f"  ✅ 成功: {success_count}件")
print(f"  ⚠️  スキップ: {skip_count}件")
print(f"  ❌ エラー: {error_count}件")
print(f"  📁 保存先: ../../data/graphs/page_task_combined/{{Bright|Dark}}/{{SubjectID}}/")
print("=" * 80)

# 相関係数分析（カテゴリ別）

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os
from datetime import datetime

def calculate_correlation_by_category(file_path, target_col, output_dir, condition_name):
    """
    カテゴリと領域ごとの相関係数表を作成
    
    Parameters:
    -----------
    file_path : str
        Excelファイルパス
    target_col : str
        目的変数のカラム名
    output_dir : str
        出力ディレクトリ
    condition_name : str
        条件名（bright/dark）
    """
    
    # カテゴリ定義
    categories = {
        '輝度': ['bL_', 'bnL_'],
        'コントラスト': ['c_'],
        'シャープネス': ['sh_'],
        '彩度': ['sa_']
    }
    
    # 領域定義
    regions = ['center', 'parafovea', 'periphery', 'all']
    
    print(f"\n{'='*80}")
    print(f"条件: {condition_name.upper()}")
    print(f"ファイル: {file_path}")
    print(f"目的変数: {target_col}")
    print(f"{'='*80}")
    
    # データ読み込み
    df = pd.read_excel(file_path)
    print(f"データ形状: {df.shape}")
    
    # 目的変数の確認
    if target_col not in df.columns:
        print(f"エラー: {target_col} がデータに存在しません")
        return
    
    # 出力ディレクトリ作成
    os.makedirs(output_dir, exist_ok=True)
    
    for category_name, keywords in categories.items():
        for region in regions:
            # カテゴリと領域に該当するカラムを検索
            category_cols = []
            for col in df.columns:
                if col == target_col:
                    continue
                # 領域のプレフィックスをチェック
                if col.startswith(f"{region}_"):
                    # カテゴリのキーワードが含まれるかチェック
                    if any(keyword in col for keyword in keywords):
                        category_cols.append(col)
            
            if len(category_cols) == 0:
                continue
            
            # 相関係数を計算
            correlations = []
            for col in category_cols:
                try:
                    # 欠損値を除外
                    valid_data = df[[target_col, col]].dropna()
                    if len(valid_data) < 10:
                        continue
                    
                    # Pearson相関係数
                    corr = valid_data[target_col].corr(valid_data[col])
                    
                    # Spearman相関係数
                    spearman_corr, spearman_pval = stats.spearmanr(valid_data[target_col], valid_data[col])
                    
                    correlations.append({
                        '特徴量': col,
                        'Pearson相関係数': corr,
                        'Spearman相関係数': spearman_corr,
                        'Spearman p値': spearman_pval,
                        'サンプル数': len(valid_data)
                    })
                except Exception as e:
                    print(f"  エラー ({col}): {str(e)}")
                    continue
            
            if len(correlations) == 0:
                continue
            
            # DataFrameに変換
            corr_df = pd.DataFrame(correlations)
            
            # Pearson相関の絶対値でソート
            corr_df = corr_df.sort_values('Pearson相関係数', key=lambda x: x.abs(), ascending=False)
            
            # CSV保存
            csv_path = os.path.join(output_dir, f"correlation_{region}_{category_name}_{target_col}_{condition_name}.csv")
            corr_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"\n{region}_{category_name}: {len(corr_df)}個の特徴量 → {csv_path}")
            
            # 上位5個を表示
            print(f"  【トップ5】")
            for idx, row in corr_df.head(5).iterrows():
                print(f"    {row['特徴量']}: Pearson={row['Pearson相関係数']:.4f}, Spearman={row['Spearman相関係数']:.4f}")
    
    print(f"\n{'='*80}")
    print(f"{condition_name.upper()} 完了")
    print(f"{'='*80}\n")

In [ ]:
# データファイルのパス設定（reduced以外のファイルを指定）
data_configs = [
    {
        "name": "dark",
        "path": "../../../penstone/data_pupil/final_2025_dark_pupil/darkfinal_recalculated_pupil_bcss_with_roi.xlsx"  # ここを実際のファイル名に変更
    },
    {
        "name": "bright",
        "path": "../../../penstone/data_pupil/final_2025_bright_pupil/final_recalculated_pupil_bcss_with_roi_withoutNan.xlsx"  # ここを実際のファイル名に変更
              }
]

# 目的変数
target_col = "corrected_pupil"

# 出力ディレクトリ
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_dir = f"correlation_analysis_{timestamp}"

# 各条件について相関分析
for config in data_configs:
    condition_name = config["name"]
    file_path = config["path"]
    output_dir = os.path.join(base_output_dir, condition_name)
    
    calculate_correlation_by_category(file_path, target_col, output_dir, condition_name)

print("\n全ての分析が完了しました")
print(f"結果保存先: {base_output_dir}")

# 相関係数分析（多重共線性除去後）

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def remove_multicollinearity(df, feature_cols, vif_threshold=10.0):
    """
    VIFを用いて多重共線性を除去
    
    Parameters:
    -----------
    df : DataFrame
        データ
    feature_cols : list
        特徴量のリスト
    vif_threshold : float
        VIFの閾値（デフォルト10）
    
    Returns:
    --------
    selected_features : list
        多重共線性除去後の特徴量リスト
    """
    
    print(f"\n多重共線性除去開始（VIF閾値: {vif_threshold}）")
    print(f"初期特徴量数: {len(feature_cols)}")
    
    # 欠損値を含む行を除外
    df_clean = df[feature_cols].dropna()
    remaining_features = feature_cols.copy()
    
    iteration = 0
    while True:
        iteration += 1
        print(f"\n  イテレーション {iteration}: {len(remaining_features)}個の特徴量")
        
        # 定数項を追加してVIFを計算
        vif_data = pd.DataFrame()
        vif_data["特徴量"] = remaining_features
        
        vif_values = []
        for i, col in enumerate(remaining_features):
            try:
                vif = variance_inflation_factor(df_clean[remaining_features].values, i)
                vif_values.append(vif)
            except:
                vif_values.append(np.inf)
        
        vif_data["VIF"] = vif_values
        vif_data = vif_data.sort_values("VIF", ascending=False)
        
        # 最大VIFを表示
        max_vif = vif_data.iloc[0]["VIF"]
        max_vif_feature = vif_data.iloc[0]["特徴量"]
        print(f"    最大VIF: {max_vif:.2f} ({max_vif_feature})")
        
        # VIF閾値を超える場合は削除
        if max_vif > vif_threshold:
            remaining_features.remove(max_vif_feature)
            print(f"    → 削除: {max_vif_feature}")
            df_clean = df_clean[remaining_features]
        else:
            print(f"    → 終了条件達成（全てVIF < {vif_threshold}）")
            break
        
        # 無限ループ防止
        if len(remaining_features) <= 1:
            print(f"    → 特徴量が1個以下になったため終了")
            break
    
    print(f"\n多重共線性除去完了")
    print(f"  削除された特徴量数: {len(feature_cols) - len(remaining_features)}")
    print(f"  残存特徴量数: {len(remaining_features)}")
    
    return remaining_features

def calculate_correlation_with_vif(file_path, target_col, output_dir, condition_name, vif_threshold=10.0):
    """
    多重共線性除去後の相関係数表を作成（領域別）
    
    Parameters:
    -----------
    file_path : str
        Excelファイルパス
    target_col : str
        目的変数のカラム名
    output_dir : str
        出力ディレクトリ
    condition_name : str
        条件名（bright/dark）
    vif_threshold : float
        VIF閾値
    """
    
    # カテゴリ定義
    categories = {
        '輝度': ['bL_', 'bnL_'],
        'コントラスト': ['c_'],
        'シャープネス': ['sh_'],
        '彩度': ['sa_']
    }
    
    # 領域定義
    regions = ['center', 'parafovea', 'periphery', 'all']
    
    print(f"\n{'='*80}")
    print(f"条件: {condition_name.upper()} (多重共線性除去後)")
    print(f"ファイル: {file_path}")
    print(f"目的変数: {target_col}")
    print(f"{'='*80}")
    
    # データ読み込み
    df = pd.read_excel(file_path)
    print(f"データ形状: {df.shape}")
    
    # 目的変数の確認
    if target_col not in df.columns:
        print(f"エラー: {target_col} がデータに存在しません")
        return
    
    # 出力ディレクトリ作成
    os.makedirs(output_dir, exist_ok=True)
    
    for category_name, keywords in categories.items():
        for region in regions:
            # カテゴリと領域に該当するカラムを検索
            category_cols = []
            for col in df.columns:
                if col == target_col:
                    continue
                # 領域のプレフィックスをチェック
                if col.startswith(f"{region}_"):
                    # カテゴリのキーワードが含まれるかチェック
                    if any(keyword in col for keyword in keywords):
                        category_cols.append(col)
            
            if len(category_cols) == 0:
                continue
            
            print(f"\n{'='*60}")
            print(f"領域: {region}, カテゴリ: {category_name}")
            print(f"{'='*60}")
            
            # 多重共線性除去
            selected_features = remove_multicollinearity(df, category_cols, vif_threshold)
            
            if len(selected_features) == 0:
                print(f"\n{region}_{category_name}: 多重共線性除去後に特徴量が残りませんでした")
                continue
            
            # 相関係数を計算
            correlations = []
            for col in selected_features:
                try:
                    # 欠損値を除外
                    valid_data = df[[target_col, col]].dropna()
                    if len(valid_data) < 10:
                        continue
                    
                    # Pearson相関係数
                    corr = valid_data[target_col].corr(valid_data[col])
                    
                    # Spearman相関係数
                    spearman_corr, spearman_pval = stats.spearmanr(valid_data[target_col], valid_data[col])
                    
                    correlations.append({
                        '特徴量': col,
                        'Pearson相関係数': corr,
                        'Spearman相関係数': spearman_corr,
                        'Spearman p値': spearman_pval,
                        'サンプル数': len(valid_data)
                    })
                except Exception as e:
                    print(f"  エラー ({col}): {str(e)}")
                    continue
            
            if len(correlations) == 0:
                print(f"\n{region}_{category_name}: 相関計算失敗")
                continue
            
            # DataFrameに変換
            corr_df = pd.DataFrame(correlations)
            
            # Pearson相関の絶対値でソート
            corr_df = corr_df.sort_values('Pearson相関係数', key=lambda x: x.abs(), ascending=False)
            
            # CSV保存
            csv_path = os.path.join(output_dir, f"correlation_VIF_{region}_{category_name}_{target_col}_{condition_name}.csv")
            corr_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
            print(f"\n結果保存: {len(corr_df)}個の特徴量 → {csv_path}")
            
            # 上位5個を表示
            print(f"\n  【トップ5】")
            for idx, row in corr_df.head(5).iterrows():
                print(f"    {row['特徴量']}: Pearson={row['Pearson相関係数']:.4f}, Spearman={row['Spearman相関係数']:.4f}")
    
    print(f"\n{'='*80}")
    print(f"{condition_name.upper()} 完了（多重共線性除去後）")
    print(f"{'='*80}\n")

In [ ]:
# データファイルのパス設定（同じファイルを使用）
data_configs_vif = [
    {
        "name": "dark",
        "path": "../../../penstone/data_pupil/final_2025_dark_pupil/darkfinal_recalculated_pupil_bcss_with_roi.xlsx"
    },
    {
        "name": "bright",
        "path": "../../../penstone/data_pupil/final_2025_bright_pupil/final_recalculated_pupil_bcss_with_roi_withoutNan.xlsx"
    }
]

# 目的変数
target_col = "corrected_pupil"

# 出力ディレクトリ
timestamp_vif = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_dir_vif = f"correlation_analysis_VIF_{timestamp_vif}"

# VIF閾値（10以上で多重共線性が高いとされる）
vif_threshold = 10.0

# 各条件について多重共線性除去後の相関分析
for config in data_configs_vif:
    condition_name = config["name"]
    file_path = config["path"]
    output_dir = os.path.join(base_output_dir_vif, condition_name)
    
    calculate_correlation_with_vif(file_path, target_col, output_dir, condition_name, vif_threshold)

print("\n全ての分析が完了しました（多重共線性除去後）")
print(f"結果保存先: {base_output_dir_vif}")

# 相関係数分析（ALL特徴量のみ・カテゴリ別）

In [ ]:
def calculate_correlation_all_features_only(file_path, target_col, output_dir, condition_name):
    """
    all_が含まれる特徴量のみを対象に、カテゴリごとの相関係数表を作成
    
    Parameters:
    -----------
    file_path : str
        Excelファイルパス
    target_col : str
        目的変数のカラム名
    output_dir : str
        出力ディレクトリ
    condition_name : str
        条件名（bright/dark）
    """
    
    # カテゴリ定義（all_が含まれるもののみ）
    categories = {
        '輝度': ['all_bL_', 'all_bnL_'],
        'コントラスト': ['all_c_'],
        'シャープネス': ['all_sh_'],
        '彩度': ['all_sa_']
    }
    
    print(f"\n{'='*80}")
    print(f"条件: {condition_name.upper()} (ALL特徴量のみ)")
    print(f"ファイル: {file_path}")
    print(f"目的変数: {target_col}")
    print(f"{'='*80}")
    
    # データ読み込み
    df = pd.read_excel(file_path)
    print(f"データ形状: {df.shape}")
    
    # 目的変数の確認
    if target_col not in df.columns:
        print(f"エラー: {target_col} がデータに存在しません")
        return
    
    # 出力ディレクトリ作成
    os.makedirs(output_dir, exist_ok=True)
    
    for category_name, keywords in categories.items():
        # カテゴリに該当するカラムを検索（all_が含まれるもののみ）
        category_cols = []
        for col in df.columns:
            if col == target_col:
                continue
            # all_が含まれ、かつカテゴリのキーワードが含まれるもの
            if 'all_' in col and any(keyword in col for keyword in keywords):
                category_cols.append(col)
        
        if len(category_cols) == 0:
            print(f"\n{category_name}: 該当カラムなし")
            continue
        
        # 相関係数を計算
        correlations = []
        for col in category_cols:
            try:
                # 欠損値を除外
                valid_data = df[[target_col, col]].dropna()
                if len(valid_data) < 10:
                    continue
                
                # Pearson相関係数
                corr = valid_data[target_col].corr(valid_data[col])
                
                # Spearman相関係数
                spearman_corr, spearman_pval = stats.spearmanr(valid_data[target_col], valid_data[col])
                
                correlations.append({
                    '特徴量': col,
                    'Pearson相関係数': corr,
                    'Spearman相関係数': spearman_corr,
                    'Spearman p値': spearman_pval,
                    'サンプル数': len(valid_data)
                })
            except Exception as e:
                print(f"  エラー ({col}): {str(e)}")
                continue
        
        if len(correlations) == 0:
            print(f"\n{category_name}: 相関計算失敗")
            continue
        
        # DataFrameに変換
        corr_df = pd.DataFrame(correlations)
        
        # Pearson相関の絶対値でソート
        corr_df = corr_df.sort_values('Pearson相関係数', key=lambda x: x.abs(), ascending=False)
        
        # CSV保存
        csv_path = os.path.join(output_dir, f"correlation_ALL_{category_name}_{target_col}_{condition_name}.csv")
        corr_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\n{category_name}: {len(corr_df)}個の特徴量 → {csv_path}")
        
        # 上位10個を表示
        print(f"  【トップ10】")
        for idx, row in corr_df.head(10).iterrows():
            print(f"    {row['特徴量']}: Pearson={row['Pearson相関係数']:.4f}, Spearman={row['Spearman相関係数']:.4f}")
    
    print(f"\n{'='*80}")
    print(f"{condition_name.upper()} 完了（ALL特徴量のみ）")
    print(f"{'='*80}\n")

In [ ]:
# データファイルのパス設定（同じファイルを使用）
data_configs_all = [
    {
        "name": "dark",
        "path": "../../../penstone/data_pupil/final_2025_dark_pupil/darkfinal_recalculated_pupil_bcss_roi_global_area_pupil.xlsx"    
    },
    {
        "name": "bright",
        "path": "../../../penstone/data_pupil/final_2025_bright_pupil/final_recalculated_pupil_bcss_with_roi_global_withoutNan_with_area_pupil.xlsx"
    }
]


# 目的変数
target_col = "corrected_pupil"

# 出力ディレクトリ
timestamp_all = datetime.now().strftime("%Y%m%d_%H%M%S")
base_output_dir_all = f"correlation_analysis_ALL_{timestamp_all}"

# 各条件についてALL特徴量のみの相関分析
for config in data_configs_all:
    condition_name = config["name"]
    file_path = config["path"]
    output_dir = os.path.join(base_output_dir_all, condition_name)
    
    calculate_correlation_all_features_only(file_path, target_col, output_dir, condition_name)

print("\n全ての分析が完了しました（ALL特徴量のみ）")
print(f"結果保存先: {base_output_dir_all}")

# 試行回数による変化分析（疲労・学習効果）
被験者ごと・試行回数ごとの縮瞳・輻輳・反応速度の変化を分析し、後半での分散増加（疲労効果）を検証する。

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['axes.unicode_minus'] = False


def analyze_trial_progression(file_path, condition_name, output_dir,
                              alphabet_only=True,
                              require_any_metric=True):
    """
    被験者ごとのみ：
      - RT / 縮瞳 / 輻輳 を 3段グラフで保存
      - trial_id を (S番号, 0..9, 1..48) の数値キーで正しく並べる
      - （任意）frontが文字(FrontIsDigit_inferred==True)だけに絞る
      - ★追加：proc列ごとに色分けしてプロット（凡例あり）
    """

    print("=" * 80)
    print(f"条件: {condition_name}")
    print(f"ファイル: {file_path}")
    print("=" * 80)

    df = pd.read_excel(file_path, engine="openpyxl")
    print(f"データ形状(元): {df.shape}")

    # ---- 列名統一（あなたのファイルに合わせる）----
    rename_map = {
        "folder_name": "subject",
        "Reaction_Time": "RT",
        "pupil_both_change_rate_mean": "miosis_rate",
        "diopter_delta": "diopter",
        "trial_id": "trial_id",
        "FrontIsDigit_inferred": "FrontIsDigit_inferred",
        "process": "process",  # ★追加
    }
    df = df.rename(columns=rename_map)

    # ---- 必須列チェック ----
    need = ["subject", "trial_id", "RT", "miosis_rate", "diopter", "FrontIsDigit_inferred", "process"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"必要列がありません: {missing}\n列一覧: {list(df.columns)}")

    # procの表記ゆれ対策（None/NaNを文字列に）
    df["process"] = df["process"].astype(str)

    # ---- front=文字のみ（要望）----
    if alphabet_only:
        before = len(df)
        df = df[df["FrontIsDigit_inferred"] == True].copy()
        after = len(df)
        print(f"[filter] alphabet-only: {after}/{before}")

    # ---- “全部NaNしかない”問題の対策（必要なら）----
    if require_any_metric:
        before = len(df)
        df = df[df[["RT", "miosis_rate", "diopter"]].notna().any(axis=1)].copy()
        after = len(df)
        print(f"[filter] any-metric-notna: {after}/{before}")

    # ---- trial_id を数値キーで分解して “正しい順” に並べる ----
    parts = df["trial_id"].astype(str).str.extract(r"^S(\d+)_([0-9]+)_([0-9]+)$")
    if parts.isna().any(axis=None):
        bad = df.loc[parts.isna().any(axis=1), "trial_id"].head(10).tolist()
        raise ValueError(
            "trial_id が想定形式 Sxx_0..9_1..48 に一致しないものがあります。\n"
            f"例: {bad}"
        )

    df["sid"] = parts[0].astype(int)          # S01 -> 1
    df["block"] = parts[1].astype(int)        # 0..9
    df["t_in_block"] = parts[2].astype(int)   # 1..48
    df["trial_seq"] = df["block"] * 48 + (df["t_in_block"] - 1)

    df = df.sort_values(["subject", "sid", "block", "t_in_block"]).reset_index(drop=True)

    # ---- 出力ディレクトリ ----
    subj_root = os.path.join(output_dir, condition_name, "subjects")
    os.makedirs(subj_root, exist_ok=True)
    print("保存先:", os.path.abspath(subj_root))

    subjects = sorted(df["subject"].unique())
    print(f"被験者数(残った): {len(subjects)}")

    metrics = [
        ("RT", "反応時間 (Reaction_Time)"),
        ("miosis_rate", "縮瞳 (pupil_both_change_rate_mean)"),
        ("diopter", "輻輳 (diopter_delta)"),
    ]

    saved = 0
    for subj in subjects:
        d = df[df["subject"] == subj].copy()

        n_rt = d["RT"].notna().sum()
        n_mio = d["miosis_rate"].notna().sum()
        n_dpt = d["diopter"].notna().sum()

        if (n_rt == 0) and (n_mio == 0) and (n_dpt == 0):
            print(f"[SKIP] {subj}: all NaN")
            continue

        fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

        # trial_seq -> trial_id（ラベル用）
        id_map = (
            d[["trial_seq", "trial_id"]]
            .drop_duplicates("trial_seq")
            .set_index("trial_seq")["trial_id"]
            .to_dict()
        )

        # procの並び（凡例の順）
        proc_order = sorted(d["process"].dropna().unique())

        # ★指標ごとに、proc別に描画（色は自動で変わる）
        for ax, (col, ylabel) in zip(axes, metrics):
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.3)

            for p in proc_order:
                dp = d[d["process"] == p].dropna(subset=[col]).copy()
                if len(dp) == 0:
                    continue
                ax.scatter(dp["trial_seq"], dp[col], s=16, alpha=0.8, label=p)
                ax.plot(dp["trial_seq"], dp[col], linewidth=1.0, alpha=0.6)

            # 同じ凡例が3段で邪魔なら「上段だけ」にする
            # ここでは上段だけ表示
            if ax is axes[0]:
                ax.legend(loc="best", fontsize=9, frameon=True)

        axes[0].set_title(f"{condition_name} / {subj}（trial_id順=正しい）")
        axes[-1].set_xlabel("trial（block*48 + (t-1)）")

        xticks = np.array(sorted(d["trial_seq"].unique()))
        step = max(1, len(xticks) // 12)
        xt_show = xticks[::step]
        axes[-1].set_xticks(xt_show)
        axes[-1].set_xticklabels([id_map.get(int(t), str(t)) for t in xt_show],
                                 rotation=45, ha="right")

        plt.tight_layout()

        safe_subj = str(subj).replace("/", "_").replace("\\", "_")
        out_path = os.path.join(subj_root, f"{safe_subj}_RT_miosis_diopter.png")
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

        saved += 1
        print(f"[SAVE] {subj}: RT={n_rt}, miosis={n_mio}, diopter={n_dpt}")

    print(f"保存完了: {saved} subjects → {os.path.abspath(subj_root)}")
    return df


if __name__ == "__main__":
    base_dir = "../../../penstone_2025/data/log_with_emr_metrics"
    params = "lag0p5_mioF10_BLstim120"
    n = 15

    bright_file = f"{base_dir}/{params}/merged/integrated_bright_metrics_n{n}.xlsx"
    dark_file   = f"{base_dir}/{params}/merged/integrated_dark_metrics_n{n}.xlsx"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_root = f"trial_progression_subject_only_{timestamp}"

    if os.path.exists(bright_file):
        analyze_trial_progression(bright_file, "bright", out_root,
                                  alphabet_only=True,
                                  require_any_metric=True)
    else:
        print("Bright not found:", bright_file)

    if os.path.exists(dark_file):
        analyze_trial_progression(dark_file, "dark", out_root,
                                  alphabet_only=True,
                                  require_any_metric=True)
    else:
        print("Dark not found:", dark_file)


条件: bright
ファイル: ../../../penstone_2025/data/log_with_emr_metrics/lag0p5_mioF10_BLstim120/merged/integrated_bright_metrics_n15.xlsx
データ形状(元): (7200, 63)
[filter] alphabet-only: 2400/7200
[filter] any-metric-notna: 2399/2400
保存先: c:\Users\ryuno\Documents\penstone_2025\analysis\main_process\trial_progression_subject_only_20260119_152711\bright\subjects
被験者数(残った): 15
[SAVE] S01: RT=160, miosis=81, diopter=84
[SAVE] S02: RT=160, miosis=13, diopter=15
[SAVE] S03: RT=159, miosis=45, diopter=47
[SAVE] S04: RT=160, miosis=142, diopter=147
[SAVE] S05: RT=160, miosis=141, diopter=143
[SAVE] S06: RT=160, miosis=152, diopter=159
[SAVE] S07: RT=160, miosis=28, diopter=29
[SAVE] S08: RT=160, miosis=155, diopter=159
[SAVE] S09: RT=160, miosis=153, diopter=159
[SAVE] S10: RT=160, miosis=155, diopter=159
[SAVE] S11: RT=160, miosis=54, diopter=58
[SAVE] S12: RT=160, miosis=157, diopter=159
[SAVE] S13: RT=160, miosis=151, diopter=155
[SAVE] S14: RT=160, miosis=157, diopter=159
[SAVE] S15: RT=160, miosis=

In [ ]:
# -*- coding: utf-8 -*-
"""
anova_filtered_subjects_z_iqr_5panel_sig.py
------------------------------------------
要件（あなたの最新要求を反映）:
- trial単位のデータから
  - 前処理（列名統一 / image_key抽出 / proc限定 / FrontIsDigit_inferred==True など）
  - 被験者内 IQR 外れ値除去 → その後に被験者内 z-score 標準化（z_列を作る）  ★重要
- 被験者フィルタ:
  - z_miosis_rate の ALL（image_key無視）平均で brightonly < model の subject だけ残す
- 解析:
  - One-way ANOVA（=独立群扱い; 行をそのまま使うのでサンプル数減らしにくい）
    - 多重比較: Tukey HSD（p-adjが有意のペアのみ括弧＋p）
  - RM-ANOVA（=対応あり; complete subjectのみなのでサンプル数減りやすい）
    - 多重比較: 対応t（3ペア）+ Holm補正（補正後p<0.05のみ括弧＋p）
- グラフ:
  - image_key 4枚 + ALL の 5枚を横に並べた 1枚の画像（1×5）
  - df2理由は書かない（F(df1,df2)のみ）
- 出力フォルダ:
  <out_root>/<condition>/oneway/figures_5panel/
  <out_root>/<condition>/oneway/csv/
  <out_root>/<condition>/rm/figures_5panel/
  <out_root>/<condition>/rm/csv/

注意:
- FrontIsDigit_inferred==True を残す挙動になっている（あなたの現状コード踏襲）
  もし「文字だけ」にしたいなら preprocess 内を == False に変えてください。
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "MS Gothic"
plt.rcParams["axes.unicode_minus"] = False

IMAGE_KEYS = ["sun_empty", "sun_busy", "rain_empty", "rain_busy"]
PROCS = ["original", "brightonly", "model"]
PANELS = IMAGE_KEYS + ["ALL"]

# =========================
# image_key 抽出
# =========================
def extract_image_key_from_row(row, keys, cols=("filename", "Back_Image_Name_Used")):
    for c in cols:
        v = row.get(c, None)
        if pd.isna(v):
            continue
        s = str(v)
        for k in keys:
            if k in s:
                return k
    return None


# =========================
# 前処理（最小）
# =========================
def preprocess_minimum(df_raw):
    df = df_raw.copy()

    rename_map = {
        "folder_name": "subject",
        "process": "proc",
        "pupil_both_change_rate_mean": "miosis_rate",
        "diopter_delta": "diopter",
        "Reaction_Time": "RT",
        "trial_id": "trial_id",
    }
    df = df.rename(columns=rename_map)

    need = ["subject", "proc", "RT", "miosis_rate", "diopter"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"必要列がありません: {miss}\n列一覧: {list(df.columns)}")

    # frontフィルタ（現状踏襲: Trueを残す）
    if "FrontIsDigit_inferred" in df.columns:
        df = df[df["FrontIsDigit_inferred"] == True].copy()

    df["proc"] = df["proc"].astype(str)
    df = df[df["proc"].isin(PROCS)].copy()

    df["image_key"] = df.apply(lambda r: extract_image_key_from_row(r, IMAGE_KEYS), axis=1)

    df = df[["subject", "proc", "image_key", "RT", "miosis_rate", "diopter"]].copy()
    df = df.dropna(subset=["subject", "proc"])

    return df


# =========================
# IQR外れ値除去 → z-score（被験者内）
# =========================
def iqr_filter_and_zscore(df, metric, iqr_k=1.5, min_points=5):
    """
    被験者内で
      1) IQR外れ値除去（metric単体）
      2) 残った点で平均・SDを計算して z-score を付与
    z_<metric> を作る（外れ値や計算不能点は NaN のまま）
    """
    df = df.copy()
    zcol = f"z_{metric}"
    df[zcol] = np.nan

    for subj, g in df.groupby("subject"):
        x = g[metric].astype(float)

        x_valid = x.dropna()
        if x_valid.shape[0] < min_points:
            continue

        q1 = x_valid.quantile(0.25)
        q3 = x_valid.quantile(0.75)
        iqr = q3 - q1

        # iqr=0 の場合は外れ値フィルタを実質無効化（全て同値など）
        if not np.isfinite(iqr) or iqr == 0:
            x_filt = x_valid
        else:
            lo = q1 - iqr_k * iqr
            hi = q3 + iqr_k * iqr
            x_filt = x_valid[(x_valid >= lo) & (x_valid <= hi)]

        if x_filt.shape[0] < 2:
            continue

        sd = x_filt.std(ddof=1)
        if (not np.isfinite(sd)) or sd == 0:
            continue

        z = (x_filt - x_filt.mean()) / sd
        df.loc[z.index, zcol] = z

    return df


def add_zscores_after_iqr(df, metrics=("RT", "miosis_rate", "diopter"), iqr_k=1.5):
    df = df.copy()
    for m in metrics:
        if m in df.columns:
            df = iqr_filter_and_zscore(df, m, iqr_k=iqr_k, min_points=5)
    return df


# =========================
# 被験者フィルタ（z値で brightonly < model）
# =========================
def select_subjects_brightonly_lt_model(df, zmetric):
    pivot = df.groupby(["subject", "proc"])[zmetric].mean().unstack("proc")
    pivot = pivot.dropna(subset=["brightonly", "model"], how="any")
    ok = pivot[pivot["brightonly"] < pivot["model"]].index.tolist()
    return ok


# =========================
# Holm補正
# =========================
def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)

    for rank, idx in enumerate(order):
        adj[idx] = (m - rank) * pvals[idx]

    # 単調化
    adj_sorted = adj[order]
    for i in range(1, m):
        adj_sorted[i] = max(adj_sorted[i], adj_sorted[i - 1])
    adj[order] = adj_sorted

    return np.clip(adj, 0, 1)


# =========================
# 有意差カギカッコ
# =========================
def add_sig_brackets(ax, pairs, y_start):
    """
    pairs: list of (x1, x2, p_adj) ; 有意のものだけ渡す
    """
    if not pairs:
        return
    step = 0.15
    h = 0.06
    for i, (x1, x2, p) in enumerate(pairs):
        y = y_start + step * i
        ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.2, c="black")
        ax.text((x1 + x2) / 2, y + h + 0.02, f"p={p:.3g}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")


# =========================
# One-way ANOVA + Tukey
# =========================
def run_oneway_and_tukey(sub, zcol, min_n_per_group=3):
    df = sub.dropna(subset=["proc", zcol]).copy()
    df = df[df["proc"].isin(PROCS)].copy()
    if len(df) == 0:
        return None

    groups = [df[df["proc"] == p][zcol].astype(float).values for p in PROCS]
    ns = [len(g) for g in groups]
    if any(n < min_n_per_group for n in ns):
        return None

    vars_ = [np.nanvar(g, ddof=1) for g in groups]
    if all((v == 0) or np.isnan(v) for v in vars_):
        return None

    F, p = stats.f_oneway(*groups)
    if (not np.isfinite(F)) or (not np.isfinite(p)):
        return None

    # eta^2
    all_data = df[zcol].values.astype(float)
    grand = np.mean(all_data)
    ss_total = np.sum((all_data - grand) ** 2)
    ss_between = sum(len(g) * (np.mean(g) - grand) ** 2 for g in groups)
    eta_sq = ss_between / ss_total if ss_total > 0 else np.nan

    # Tukey（多重比較）
    try:
        tukey = pairwise_tukeyhsd(df[zcol], df["proc"], alpha=0.05)
    except Exception:
        tukey = None

    k = len(PROCS)
    df1 = k - 1
    df2 = len(df) - k

    return {
        "F": float(F),
        "p": float(p),
        "df1": float(df1),
        "df2": float(df2),
        "eta_sq": float(eta_sq),
        "ns": dict(zip(PROCS, ns)),
        "tukey": tukey,
        "df_used": df,
    }


# =========================
# RM-ANOVA + paired t + Holm
# =========================
def run_rm_anova_and_pairwise(sub, zcol):
    df = sub.dropna(subset=["subject", "proc", zcol]).copy()
    df = df[df["proc"].isin(PROCS)].copy()
    if len(df) == 0:
        return None

    counts = df.groupby("subject")["proc"].nunique()
    complete = counts[counts == len(PROCS)].index
    df = df[df["subject"].isin(complete)].copy()
    if df["subject"].nunique() < 2:
        return None

    # subject×proc 平均（1被験者1水準1値）
    agg = df.groupby(["subject", "proc"])[zcol].mean().reset_index()

    try:
        aovrm = AnovaRM(agg, depvar=zcol, subject="subject", within=["proc"])
        res = aovrm.fit()
        table = res.anova_table
        f = float(table.loc["proc", "F Value"])
        p = float(table.loc["proc", "Pr > F"])
        df1 = float(table.loc["proc", "Num DF"])
        df2 = float(table.loc["proc", "Den DF"])
        eta_p2 = (f * df1) / (f * df1 + df2) if (f * df1 + df2) > 0 else np.nan
    except Exception:
        return None

    pivot = agg.pivot(index="subject", columns="proc", values=zcol)

    # 3ペアの対応t
    pair_labels = []
    raw_p = []
    for i in range(len(PROCS)):
        for j in range(i + 1, len(PROCS)):
            p1, p2 = PROCS[i], PROCS[j]
            x = pivot[p1].values
            y = pivot[p2].values
            t, p_pair = stats.ttest_rel(x, y)
            pair_labels.append((p1, p2))
            raw_p.append(float(p_pair) if np.isfinite(p_pair) else np.nan)

    raw_p = np.array(raw_p, dtype=float)
    valid = np.isfinite(raw_p)
    adj_p = np.full_like(raw_p, np.nan)
    if valid.sum() > 0:
        adj_p[valid] = holm_adjust(raw_p[valid])

    # (proc_index1, proc_index2, p_adj)
    pair_results = []
    for (p1, p2), p_adj in zip(pair_labels, adj_p):
        i1, i2 = PROCS.index(p1), PROCS.index(p2)
        pair_results.append((i1, i2, float(p_adj) if np.isfinite(p_adj) else np.nan))

    return {
        "F": f,
        "p": p,
        "df1": df1,
        "df2": df2,
        "eta_p2": eta_p2,
        "n_subjects": int(pivot.shape[0]),
        "agg": agg,
        "pairwise_adj": pair_results,
    }


# =========================
# 5panel 描画（One-way）
# =========================
def plot_5panel_oneway(df_condition, zcol, condition_name, metric_name, out_png):
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.8), sharey=True)
    fig.suptitle(f"{condition_name} / {metric_name} / One-way ANOVA + Tukey (z, IQR後)",
                 fontsize=14, fontweight="bold")

    rows = []

    for ax, key in zip(axes, PANELS):
        if key == "ALL":
            sub = df_condition.copy()
        else:
            sub = df_condition[df_condition["image_key"] == key].dropna(subset=["image_key"]).copy()

        res = run_oneway_and_tukey(sub, zcol)
        if res is None:
            ax.text(0.5, 0.5, "Insufficient Data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(key, fontsize=10, fontweight="bold")
            ax.set_xticks(np.arange(len(PROCS)))
            ax.set_xticklabels(PROCS, rotation=15, ha="right")
            ax.grid(axis="y", alpha=0.25)
            continue

        df_used = res["df_used"]
        means = df_used.groupby("proc")[zcol].mean().reindex(PROCS)
        sems  = df_used.groupby("proc")[zcol].sem().reindex(PROCS)
        ns    = df_used.groupby("proc")[zcol].count().reindex(PROCS)

        x = np.arange(len(PROCS))
        ax.bar(x, means.values, yerr=sems.values, capsize=5, alpha=0.85)
        ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.grid(axis="y", alpha=0.25)
        ax.set_xticks(x)
        ax.set_xticklabels(PROCS, rotation=15, ha="right")

        sig = "***" if res["p"] < 0.001 else "**" if res["p"] < 0.01 else "*" if res["p"] < 0.05 else ""
        title = f"{key}\nF({int(res['df1'])},{int(res['df2'])})={res['F']:.2f}, p={res['p']:.3g}{sig}\nη²={res['eta_sq']:.3f}"
        ax.set_title(title, fontsize=10, fontweight="bold")

        # Tukey 有意ペアのみ
        sig_pairs = []
        if res["tukey"] is not None:
            for row in res["tukey"].summary().data[1:]:
                g1, g2, _, p_adj, *_ = row
                reject = row[-1]
                if bool(reject):
                    i1 = PROCS.index(str(g1))
                    i2 = PROCS.index(str(g2))
                    sig_pairs.append((i1, i2, float(p_adj)))

        if sig_pairs:
            y_max = float(np.nanmax(means.values + sems.values))
            add_sig_brackets(ax, sig_pairs, y_start=y_max + 0.12)

        rows.append({
            "condition": condition_name, "metric": metric_name, "zcol": zcol, "image_key": key,
            "F": res["F"], "p": res["p"], "df1": res["df1"], "df2": res["df2"], "eta_sq": res["eta_sq"],
            "n_total": int(ns.sum()),
            "n_original": int(ns.get("original", 0)),
            "n_brightonly": int(ns.get("brightonly", 0)),
            "n_model": int(ns.get("model", 0)),
        })

    axes[0].set_ylabel(f"z_{metric_name}", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.close(fig)

    return pd.DataFrame(rows)


# =========================
# 5panel 描画（RM）
# =========================
def plot_5panel_rm(df_condition, zcol, condition_name, metric_name, out_png):
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.8), sharey=True)
    fig.suptitle(f"{condition_name} / {metric_name} / RM-ANOVA + paired t (Holm) (z, IQR後)",
                 fontsize=14, fontweight="bold")

    rows = []

    for ax, key in zip(axes, PANELS):
        if key == "ALL":
            sub = df_condition.copy()
        else:
            sub = df_condition[df_condition["image_key"] == key].dropna(subset=["image_key"]).copy()

        res = run_rm_anova_and_pairwise(sub, zcol)
        if res is None:
            ax.text(0.5, 0.5, "RM-ANOVA N/A", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(key, fontsize=10, fontweight="bold")
            ax.set_xticks(np.arange(len(PROCS)))
            ax.set_xticklabels(PROCS, rotation=15, ha="right")
            ax.grid(axis="y", alpha=0.25)
            continue

        agg = res["agg"]
        means = agg.groupby("proc")[zcol].mean().reindex(PROCS)
        sems  = agg.groupby("proc")[zcol].sem().reindex(PROCS)

        x = np.arange(len(PROCS))
        ax.bar(x, means.values, yerr=sems.values, capsize=5, alpha=0.85)
        ax.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.grid(axis="y", alpha=0.25)
        ax.set_xticks(x)
        ax.set_xticklabels(PROCS, rotation=15, ha="right")

        sig = "***" if res["p"] < 0.001 else "**" if res["p"] < 0.01 else "*" if res["p"] < 0.05 else ""
        title = f"{key}\nF({int(res['df1'])},{int(res['df2'])})={res['F']:.2f}, p={res['p']:.3g}{sig}\nηp²={res['eta_p2']:.3f}, N={res['n_subjects']}"
        ax.set_title(title, fontsize=10, fontweight="bold")

        # Holm補正後pで有意なペアのみ
        sig_pairs = []
        for (i1, i2, p_adj) in res["pairwise_adj"]:
            if np.isfinite(p_adj) and (p_adj < 0.05):
                sig_pairs.append((i1, i2, float(p_adj)))

        if sig_pairs:
            y_max = float(np.nanmax(means.values + sems.values))
            add_sig_brackets(ax, sig_pairs, y_start=y_max + 0.12)

        rows.append({
            "condition": condition_name, "metric": metric_name, "zcol": zcol, "image_key": key,
            "F": res["F"], "p": res["p"], "df1": res["df1"], "df2": res["df2"],
            "eta_p2": res["eta_p2"], "n_subjects_complete": res["n_subjects"],
        })

    axes[0].set_ylabel(f"z_{metric_name}", fontsize=11)
    plt.tight_layout()
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.close(fig)

    return pd.DataFrame(rows)


# =========================
# 条件解析
# =========================
def analyze_condition(file_path, condition_name, out_dir,
                      metric_list=("RT", "miosis_rate", "diopter"),
                      filter_metric_for_subjects="miosis_rate",
                      iqr_k=1.5):
    os.makedirs(out_dir, exist_ok=True)

    oneway_dir = os.path.join(out_dir, "oneway")
    rm_dir = os.path.join(out_dir, "rm")
    for d in [oneway_dir, rm_dir]:
        os.makedirs(os.path.join(d, "figures_5panel"), exist_ok=True)
        os.makedirs(os.path.join(d, "csv"), exist_ok=True)

    df_raw = pd.read_excel(file_path, engine="openpyxl")
    df = preprocess_minimum(df_raw)

    # IQR外れ値除去→z化（被験者内）
    df = add_zscores_after_iqr(df, metrics=metric_list, iqr_k=iqr_k)

    # 解析に使うのは z が付いた行だけ（フィルタ対象指標のzがNaNは落とす）
    z_filter = f"z_{filter_metric_for_subjects}"
    if z_filter not in df.columns:
        raise ValueError(f"{z_filter} が作られていません。列: {list(df.columns)}")

    # 被験者フィルタ（z値で brightonly < model）
    ok_subjects = select_subjects_brightonly_lt_model(df, zmetric=z_filter)
    df = df[df["subject"].isin(ok_subjects)].copy()

    print(f"[{condition_name}] subjects with brightonly < model by {z_filter} (ALL mean): {len(ok_subjects)}")

    if len(df) == 0:
        print(f"[{condition_name}] データなし（フィルタ後）")
        return

    # metricごとに 5panel 画像を作る（z列使用）
    for metric in metric_list:
        zcol = f"z_{metric}"
        if zcol not in df.columns:
            print(f"[{condition_name}] skip {metric}: {zcol} not found")
            continue

        # --- One-way 5panel ---
        out_png_ow = os.path.join(oneway_dir, "figures_5panel", f"{condition_name}_{metric}_oneway_5panel.png")
        df_ow = plot_5panel_oneway(df, zcol, condition_name, metric, out_png_ow)
        df_ow.to_csv(os.path.join(oneway_dir, "csv", f"{condition_name}_{metric}_oneway.csv"),
                     index=False, encoding="utf-8-sig")
        print(f"  [SAVE] {out_png_ow}")

        # --- RM 5panel ---
        out_png_rm = os.path.join(rm_dir, "figures_5panel", f"{condition_name}_{metric}_rm_5panel.png")
        df_rm = plot_5panel_rm(df, zcol, condition_name, metric, out_png_rm)
        df_rm.to_csv(os.path.join(rm_dir, "csv", f"{condition_name}_{metric}_rm.csv"),
                     index=False, encoding="utf-8-sig")
        print(f"  [SAVE] {out_png_rm}")


# =========================
# main
# =========================
if __name__ == "__main__":
    base_dir = "../../../penstone_2025/data/log_with_emr_metrics"
    params = "lag0p5_mioF10_BLstim120"
    n = 15

    bright_file = f"{base_dir}/{params}/merged/integrated_bright_metrics_n{n}.xlsx"
    dark_file   = f"{base_dir}/{params}/merged/integrated_dark_metrics_n{n}.xlsx"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_root = f"anova_filtered_subjects_z_iqr_{timestamp}"
    os.makedirs(out_root, exist_ok=True)

    if os.path.exists(bright_file):
        analyze_condition(
            bright_file,
            "bright",
            os.path.join(out_root, "bright"),
            metric_list=("RT", "miosis_rate", "diopter"),
            filter_metric_for_subjects="miosis_rate",
            iqr_k=1.5,
        )
    else:
        print("Bright not found:", bright_file)

    if os.path.exists(dark_file):
        analyze_condition(
            dark_file,
            "dark",
            os.path.join(out_root, "dark"),
            metric_list=("RT", "miosis_rate", "diopter"),
            filter_metric_for_subjects="miosis_rate",
            iqr_k=1.5,
        )
    else:
        print("Dark not found:", dark_file)


[bright] filter brightonly<model by miosis_rate (ALL mean): 5 subjects
  [SAVE] anova_filtered_subjects_20260119_195313\bright\oneway\figures_5panel\bright_RT_oneway_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\bright\rm\figures_5panel\bright_RT_rm_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\bright\oneway\figures_5panel\bright_miosis_rate_oneway_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\bright\rm\figures_5panel\bright_miosis_rate_rm_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\bright\oneway\figures_5panel\bright_diopter_oneway_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\bright\rm\figures_5panel\bright_diopter_rm_5panel.png
[dark] filter brightonly<model by miosis_rate (ALL mean): 10 subjects
  [SAVE] anova_filtered_subjects_20260119_195313\dark\oneway\figures_5panel\dark_RT_oneway_5panel.png
  [SAVE] anova_filtered_subjects_20260119_195313\dark\rm\figures_5panel\dark_RT_rm_5panel.png
  [SAVE] anova_filtered